In [3]:
pip install transformers torch bert-extractive-summarizer datasets


Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



In [1]:
import os
from summarizer import Summarizer
from torch.utils.data import Dataset, random_split
from transformers import TrainingArguments, Trainer
import torch

# ✅ Step 1: Define Paths to Train & Test Data
train_judgement_path = r"C:\Users\kriti\Documents\Project-iml\IN-Abs\train-data\judgement"
train_summary_path = r"C:\Users\kriti\Documents\Project-iml\IN-Abs\train-data\summary"

# ✅ Step 2: Load Text Files
def load_text_files(folder_path):
    texts = {}
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        with open(file_path, "r", encoding="utf-8") as f:
            texts[filename] = f.read()
    return texts

# Load train dataset
train_judgements = load_text_files(train_judgement_path)
train_summaries = load_text_files(train_summary_path)

# ✅ Step 3: Define Extractive Summarization Dataset
class ExtractiveSummarizationDataset(Dataset):
    def __init__(self, judgements, summaries, model):
        self.judgements = list(judgements.values())
        self.summaries = list(summaries.values())
        self.model = model

    def __len__(self):
        return len(self.judgements)

    def __getitem__(self, idx):
        input_text = self.judgements[idx]

        # Generate extractive summary using the BERT-based model
        extractive_summary = self.model(input_text, ratio=0.2)  # Extracts 20% of important sentences

        return {
            "input_text": input_text,
            "generated_summary": extractive_summary,
            "reference_summary": self.summaries[idx],
        }

# ✅ Step 4: Load Extractive Summarization Model
model = Summarizer(model="bert-base-uncased")

# ✅ Step 5: Create Dataset
train_dataset = ExtractiveSummarizationDataset(train_judgements, train_summaries, model)

# ✅ Step 6: Train Model
# BERT Extractive Summarizer is **unsupervised**, so no training is required.
# Instead, we directly use it to extract key sentences.

# ✅ Step 7: Test on a Sample Judgment
sample_text = list(train_judgements.values())[0]  # Take the first judgment
generated_summary = model(sample_text, ratio=0.01)  # Extract top 20% sentences

print("Original Judgment:\n", sample_text[:500])  # Show first 500 characters
print("\nExtractive Summary:\n", generated_summary)


Original Judgment:
 Appeal No. LXVI of 1949.
Appeal from the High Court of judicature, Bombay, in a reference under section 66 of the Indian Income tax Act, 1022.
K.M. Munshi (N. P. Nathvani, with him), for the appel lant. ' M.C. Setalvad, Attorney General for India (H. J. Umrigar, with him), for the respondent. 1950.
May 26.
The judgment of the Court was delivered by MEHR CHAND MAHAJAN J.
This is an appeal against a judgment of the High Court of Judicature at Bombay in an income tax matter and it raises the questi

Extractive Summary:
 Appeal from the High Court of judicature, Bombay, in a reference under section 66 of the Indian Income tax Act, 1022. Though the decision proceeded on the principle that the outgoings were not part of the assessee 's income at all, the framers of the amending Act of 1939 wanted, apparently, to extend the principle, so far as the assessment of property was concerned, even to cases where obligatory payments had to be made out of the assessee 's income fro

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


In [2]:
pip install rouge-score


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def cosine_sim(text1, text2):
    vect = TfidfVectorizer().fit([text1, text2])
    tfidf = vect.transform([text1, text2])
    return cosine_similarity(tfidf[0:1], tfidf[1:2])[0][0]

similarity = cosine_sim(reference, generated)
print(f"Cosine Similarity: {similarity:.4f}")


Cosine Similarity: 0.7330


In [15]:
def calculate_coverage(generated_summary, reference_summary):
    ref_words = set(reference_summary.lower().split())
    gen_words = set(generated_summary.lower().split())
    if len(ref_words) == 0:
        return 0
    return len(gen_words.intersection(ref_words)) / len(ref_words)

# Example usage:
coverage = calculate_coverage(generated_summary, train_summaries[list(train_summaries.keys())[0]])
print("Coverage:", coverage)

Coverage: 0.3472222222222222


In [17]:
def calculate_compression_ratio(generated_summary, original_text):
    original_length = len(original_text.split())
    summary_length = len(generated_summary.split())
    return summary_length / original_length if original_length > 0 else 0

# Example usage:
compression = calculate_compression_ratio(generated_summary, sample_text)
print("Compression Ratio:", compression)

Compression Ratio: 0.03376777251184834


In [19]:
from rouge import Rouge
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Helper functions for evaluation metrics
def calculate_bleu(generated_summary, reference_summary):
    reference = [reference_summary.split()]
    candidate = generated_summary.split()
    smoothie = SmoothingFunction().method4
    return sentence_bleu(reference, candidate, smoothing_function=smoothie)

def calculate_content_overlap(generated_summary, reference_summary):
    vectorizer = TfidfVectorizer()
    tfidf = vectorizer.fit_transform([generated_summary, reference_summary])
    return cosine_similarity(tfidf[0:1], tfidf[1:2])[0][0]

def calculate_coverage(generated_summary, reference_summary):
    ref_words = set(reference_summary.lower().split())
    gen_words = set(generated_summary.lower().split())
    if len(ref_words) == 0:
        return 0
    return len(gen_words.intersection(ref_words)) / len(ref_words)

def calculate_compression_ratio(generated_summary, original_text):
    original_length = len(original_text.split())
    summary_length = len(generated_summary.split())
    return summary_length / original_length if original_length > 0 else 0

# Main evaluation function
def evaluate_summary(generated_summary, reference_summary, original_text):
    results = {}
    
    # ROUGE Scores
    rouge = Rouge()
    try:
        rouge_scores = rouge.get_scores(generated_summary, reference_summary)
        results.update(rouge_scores[0])
    except:
        results.update({'rouge-1': {'f': 0, 'p': 0, 'r': 0},
                       'rouge-2': {'f': 0, 'p': 0, 'r': 0},
                       'rouge-l': {'f': 0, 'p': 0, 'r': 0}})
    
    # BLEU Score
    results['bleu'] = calculate_bleu(generated_summary, reference_summary)
    
    # Content Overlap
    results['content_overlap'] = calculate_content_overlap(generated_summary, reference_summary)
    
    # Coverage
    results['coverage'] = calculate_coverage(generated_summary, reference_summary)
    
    # Compression Ratio
    results['compression_ratio'] = calculate_compression_ratio(generated_summary, original_text)
    
    return results

# Evaluate your sample
sample_reference = train_summaries[list(train_summaries.keys())[0]]
evaluation = evaluate_summary(generated_summary, sample_reference, sample_text)
print("Full Evaluation Metrics:")
for metric, score in evaluation.items():
    if isinstance(score, dict):
        print(f"{metric}:")
        for sub_metric, sub_score in score.items():
            print(f"  {sub_metric}: {sub_score:.4f}")
    else:
        print(f"{metric}: {score:.4f}")

Full Evaluation Metrics:
rouge-1:
  r: 0.3600
  p: 0.3375
  f: 0.3484
rouge-2:
  r: 0.1181
  p: 0.1429
  f: 0.1293
rouge-l:
  r: 0.3467
  p: 0.3250
  f: 0.3355
bleu: 0.0918
content_overlap: 0.6772
coverage: 0.3472
compression_ratio: 0.0338


In [24]:
import os
from summarizer import Summarizer
from sklearn.model_selection import train_test_split
from rouge import Rouge
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# 1. Load Data
def load_text_files(folder_path):
    texts = {}
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        with open(file_path, "r", encoding="utf-8") as f:
            texts[filename] = f.read()
    return texts

# 2. Initialize Model (No training needed - pretrained)
model = Summarizer(model="bert-base-uncased")

# 3. Evaluation Metrics
def evaluate_summary(generated_summary, reference_summary, original_text):
    rouge = Rouge()
    metrics = {}
    
    # ROUGE Scores
    try:
        rouge_scores = rouge.get_scores(generated_summary, reference_summary)
        metrics.update(rouge_scores[0])
    except:
        metrics.update({'rouge-1': {'f': 0, 'p': 0, 'r': 0},
                       'rouge-2': {'f': 0, 'p': 0, 'r': 0},
                       'rouge-l': {'f': 0, 'p': 0, 'r': 0}})
    
    # BLEU Score
    metrics['bleu'] = sentence_bleu(
        [reference_summary.split()],
        generated_summary.split(),
        smoothing_function=SmoothingFunction().method4
    )
    
    # Content Overlap
    vectorizer = TfidfVectorizer()
    tfidf = vectorizer.fit_transform([generated_summary, reference_summary])
    metrics['content_overlap'] = cosine_similarity(tfidf[0:1], tfidf[1:2])[0][0]
    
    # Coverage
    ref_words = set(reference_summary.lower().split())
    gen_words = set(generated_summary.lower().split())
    metrics['coverage'] = len(gen_words & ref_words) / len(ref_words) if ref_words else 0
    
    # Compression
    metrics['compression_ratio'] = len(generated_summary.split()) / len(original_text.split())
    
    return metrics

# 4. Main Pipeline
def main():
    # Load data
    judgements = load_text_files(r"C:\Users\kriti\Documents\Project-iml\IN-Abs\train-data\judgement")
    summaries = load_text_files(r"C:\Users\kriti\Documents\Project-iml\IN-Abs\train-data\summary")
    
    # Split data
    train_judgements, test_judgements, train_summaries, test_summaries = train_test_split(
        list(judgements.values()), 
        list(summaries.values()),
        test_size=0.2,
        random_state=42
    )
    
    # Evaluate on test set
    all_metrics = []
    for text, ref_summary in zip(test_judgements[:50], test_summaries[:50]):  # Test on 10 samples
        gen_summary = model(text, ratio=0.2)
        metrics = evaluate_summary(gen_summary, ref_summary, text)
        all_metrics.append(metrics)
    
    # Aggregate results
    avg_metrics = {
        'rouge-1_f': np.mean([m['rouge-1']['f'] for m in all_metrics]),
        'rouge-l_f': np.mean([m['rouge-l']['f'] for m in all_metrics]),
        'coverage': np.mean([m['coverage'] for m in all_metrics]),
        'compression': np.mean([m['compression_ratio'] for m in all_metrics])
    }
    
    print("\nAverage Performance:")
    for k, v in avg_metrics.items():
        print(f"{k}: {v:.4f}")



In [25]:
main()

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Window


Average Performance:
rouge-1_f: 0.4139
rouge-l_f: 0.3777
coverage: 0.4072
compression: 0.1760
